# Assignment 4 – Reweighing (Kamiran & Calders) on Adult, German and COMPAS

**Reweighing** is a *pre-processing* method: it leaves features and labels untouched and gives every training instance a weight so that, in the weighted training data, the protected attribute and the label are statistically independent. A classifier that supports instance weights is then trained on the weighted data. There are no hyper-parameters and nothing to optimise, so (unlike LFR) there is no tuning step and the whole pipeline runs in seconds.

Pipeline per dataset and random split:
1. Load data, define privileged / unprivileged group (single binary protected attribute)
2. **Before mitigation:** DP_Diff, DIR, consistency from the *original labels*
3. Stratified train/test split, min-max scaling (fit on train only)
4. **Baseline** classifier (no weights) vs. **Reweighing + classifier** (same classifier trained with the instance weights)
5. **After mitigation:** DP_Diff, DIR, consistency + predictive performance on the held-out test set
6. Repeat over several random splits (mean ± std); seed 42 is the headline split

**Conventions used (state these in your report):**
- Labels are recoded so that `1` = favourable outcome (matters for COMPAS, where AIF360 treats `0` = no recidivism as favourable).
- `DP_Diff = P(fav | privileged) - P(fav | unprivileged)`, `DIR = P(fav | unprivileged) / P(fav | privileged)` (as on the lecture slides). Optimal: 0 and 1.
- **Two settings:** the main experiment uses the protected attribute as one of the classifier inputs (AIF360's standard pipeline; the paper's second scenario). Because the paper's own main assumption (A2) is that the protected attribute is *not* used for prediction, the same experiment is repeated without it as a sensitivity check.
- **Extra fairness measure:** the average equalized-odds difference `AEO_Diff` from the lecture (mean of the true-positive-rate gap and the false-positive-rate gap, privileged minus unprivileged, optimum 0). Reweighing only aims at demographic parity, so AEO_Diff shows whether the error rates of the two groups also moved closer.
- Consistency uses k = 5 nearest neighbours in the *scaled, non-sensitive* feature space of the original data, so before/after values live in the same space.

**Project layout:** run this notebook from the project root (the folder that contains `loaders/`). The dataset loaders in `loaders/` are unchanged copies of AIF360's; raw data lives in `data/raw/` and is downloaded by section 2. See `README.md`.

## 0. Setup

In [1]:
# Run once if needed (see requirements.txt). NOTE: AIF360 0.6.1 does not work with pandas 3.x (dataset loaders
# crash with a StringDtype TypeError), so pandas is pinned below 3:
# %pip install -r requirements.txt
import os, sys, time, warnings, urllib.request
T_START = time.time()
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

# Local copies of AIF360's dataset loaders (the notebook must be run from the project root, next to loaders/)
try:
    HERE = os.path.dirname(os.path.abspath(__file__))     # running as a script (e.g. from IntelliJ)
except NameError:
    HERE = os.getcwd()                                    # running as a notebook
os.chdir(HERE)
if HERE not in sys.path:
    sys.path.insert(0, HERE)
from loaders.adult_dataset import AdultDataset
from loaders.german_dataset import GermanDataset
from loaders.compas_dataset import CompasDataset
from aif360.algorithms.preprocessing.reweighing import Reweighing

try:
    from IPython.display import display as show           # notebook: rendered tables
except ImportError:
    def show(x):
        print(x.to_string() if hasattr(x, "to_string") else x)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

pip install 'aif360[AdversarialDebiasing]'
pip install 'aif360[AdversarialDebiasing]'
pip install 'aif360[Reductions]'
pip install 'aif360[Reductions]'
pip install 'aif360[inFairness]'
pip install 'aif360[Reductions]'


## 1. Configuration
- `EVAL_SEEDS`: random stratified train/test splits. Reweighing is cheap, so 10 splits per dataset (also for Adult, on the full data – no subsampling needed).
- `CLASSIFIERS`: models trained with and without the weights. The default is logistic regression (same as in the LFR notebook, so results are comparable). A second model is a useful robustness check because the effect of reweighing depends on how the model uses the weights – uncomment the decision tree to add it (every classifier must accept `sample_weight`).

In [2]:
TEST_SIZE   = 0.30
KNN_K       = 5          # neighbours for consistency

DATASETS = ["German", "COMPAS", "Adult"]
USE_SENSITIVE_AS_FEATURE = True    # MAIN experiment: the protected attribute is also a classifier input (AIF360's standard
                                   # pipeline; the paper's second scenario, Fig. 7). Keep True: the variant below is the opposite.
RUN_A2_VARIANT = True              # ALSO run the paper's assumption A2: the protected attribute is used for the weights and the
                                   # evaluation but is NOT a classifier input (reported as a sensitivity table)
COMPAS_DROP_CHARGE_DESC = True   # kept identical to the LFR notebook so that results are comparable (the ~400 one-hot
                                 # columns of c_charge_desc are no speed problem for Reweighing: set False to keep them)

EVAL_SEEDS  = {n: [SEED] + list(range(1, 10)) for n in DATASETS}      # seed 42 = headline split

CLASSIFIERS = {"Logistic regression": lambda: LogisticRegression(max_iter=2000)}
# CLASSIFIERS["Decision tree (depth 6)"] = lambda: DecisionTreeClassifier(max_depth=6, random_state=SEED)

## 2. Get the raw data
The loaders in `loaders/` read the raw files from `data/raw/<dataset>/` (relative to the project root). This cell downloads them there and skips files that already exist.
If a download fails, fetch the file manually and place it in the printed folder.

In [3]:
from loaders import adult_dataset as _ad
# same location the loaders use: <folder of loaders/>/../data/raw
RAW = os.path.abspath(os.path.join(os.path.dirname(_ad.__file__), "..", "data", "raw"))
print("raw data folder:", RAW)
SOURCES = {
  "adult": {
    "adult.data":  "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data",
    "adult.test":  "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test",
    "adult.names": "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.names"},
  "german": {
    "german.data": "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data",
    "german.doc":  "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.doc"},
  "compas": {
    "compas-scores-two-years.csv": "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv"},
}
for folder, files in SOURCES.items():
    os.makedirs(os.path.join(RAW, folder), exist_ok=True)
    for fname, url in files.items():
        path = os.path.join(RAW, folder, fname)
        if os.path.exists(path):
            continue
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=60) as r:
                data = r.read()
            with open(path, "wb") as f:      # only written after a complete download
                f.write(data)
            print("downloaded", folder, fname)
        except Exception as e:
            print(f"!! could not download {fname}: {e}\n   place it manually in {os.path.join(RAW, folder)}")

raw data folder: C:\Users\arthu\IdeaProjects\datawranglingassignment4\data\raw


## 3. Datasets and group definitions
One binary protected attribute per dataset (LFR in AIF360 supports exactly one privileged and one unprivileged group):

| Dataset | Protected attribute | Privileged (=1) |
|---|---|---|
| Adult | sex | Male |
| German | sex | Male |
| COMPAS | race | Caucasian |

In [4]:
def load_dataset(name):
    if name == "Adult":
        # 'race' is no longer a protected attribute here, so it must be one-hot encoded like the other categoricals
        ds = AdultDataset(protected_attribute_names=["sex"], privileged_classes=[["Male"]],
                          categorical_features=["workclass", "education", "marital-status", "occupation",
                                                "relationship", "native-country", "race"])
        prot = "sex"
    elif name == "German":
        ds = GermanDataset(protected_attribute_names=["sex"], privileged_classes=[["male"]],
                           features_to_drop=["personal_status"])
        prot = "sex"
    elif name == "COMPAS":
        # 'sex' is no longer a protected attribute here, so it must be one-hot encoded like the other categoricals
        keep = ["sex", "age", "age_cat", "race", "juv_fel_count", "juv_misd_count", "juv_other_count",
                "priors_count", "c_charge_degree"] + ([] if COMPAS_DROP_CHARGE_DESC else ["c_charge_desc"])
        cats = ["age_cat", "c_charge_degree", "sex"] + ([] if COMPAS_DROP_CHARGE_DESC else ["c_charge_desc"])
        ds = CompasDataset(protected_attribute_names=["race"], privileged_classes=[["Caucasian"]],
                           features_to_keep=keep, categorical_features=cats)
        prot = "race"
    else:
        raise ValueError(name)
    # Recode labels so that 1 = favourable outcome, 0 = unfavourable
    y = (ds.labels.ravel() == ds.favorable_label).astype(float).reshape(-1, 1)
    ds.labels = y
    ds.favorable_label, ds.unfavorable_label = 1.0, 0.0
    return ds, prot

def prot_vector(ds, prot):
    return ds.protected_attributes[:, ds.protected_attribute_names.index(prot)].astype(int)  # 1 = privileged

## 4. Fairness and performance metrics
All three fairness metrics take a vector of favourable-outcome indicators, so the *same* function is used for the original labels (before) and for predictions (after).

In [5]:
def consistency(y, X_nonsens, k=KNN_K):
    # 1 - mean |y_i - mean(y of the k nearest neighbours of i)|   (individual fairness, optimum = 1)
    nn = NearestNeighbors(n_neighbors=k + 1).fit(X_nonsens)
    idx = nn.kneighbors(X_nonsens, return_distance=False)[:, 1:]   # drop the point itself
    return 1.0 - np.mean(np.abs(y - y[idx].mean(axis=1)))

def fairness_report(y, s, X_nonsens):
    # y: 1 = favourable; s: 1 = privileged
    p_priv, p_unpriv = y[s == 1].mean(), y[s == 0].mean()
    return {"P(fav|priv)": p_priv, "P(fav|unpriv)": p_unpriv,
            "DP_Diff": p_priv - p_unpriv,          # optimum 0
            "DIR": p_unpriv / p_priv,              # optimum 1, 'legal' range 0.8-1.25
            "Consistency": consistency(y, X_nonsens),
            "PosRate": y.mean()}                   # overall share of favourable outcomes (the paper's assumption A3)

def equalized_odds_diff(y_true, y_pred, s):
    # EXTRA MEASURE: average equalized-odds difference (lecture slide 17) = mean of the true-positive-rate gap and the
    # false-positive-rate gap, privileged minus unprivileged (optimum 0). The two gaps can cancel each other, so they
    # are also returned separately. y_true / y_pred: 1 = favourable; s: 1 = privileged.
    def rate(y_val, s_val):
        m = (y_true == y_val) & (s == s_val)
        return y_pred[m].mean() if m.any() else np.nan
    tpr_gap = rate(1, 1) - rate(1, 0)
    fpr_gap = rate(0, 1) - rate(0, 0)
    return {"AEO_Diff": 0.5 * (tpr_gap + fpr_gap), "TPR_Gap": tpr_gap, "FPR_Gap": fpr_gap}

def performance_report(y_true, y_pred):
    return {"Accuracy": accuracy_score(y_true, y_pred),
            "BalancedAcc": balanced_accuracy_score(y_true, y_pred),
            "F1": f1_score(y_true, y_pred)}

def nonsens_space(X, ds, prot):
    # drop the protected-attribute column from the (scaled) feature matrix
    j = ds.feature_names.index(prot)
    return np.delete(X, j, axis=1)

## 5. Pipeline building blocks
- `prepare(name, seed)`: stratified split on (label, protected attribute) + scaling (fit on train only).
- `reweigh(P)`: AIF360's `Reweighing`, fitted on the **training** part only; returns one weight per training instance.
- `run_dataset(name, seed)`: for every classifier, trains once without weights (**baseline**) and once with the weights (**Reweighing + classifier**), then evaluates both on the same held-out test split.

In [6]:
_DS_CACHE, _FULL_CACHE = {}, {}

def get_dataset(name):
    if name not in _DS_CACHE:
        _DS_CACHE[name] = load_dataset(name)
    return _DS_CACHE[name]

def full_data_report(name):
    # required 'before' values: original labels of the whole dataset (no split, no classifier)
    if name not in _FULL_CACHE:
        ds, prot = get_dataset(name)
        X = MinMaxScaler().fit_transform(ds.features)
        _FULL_CACHE[name] = fairness_report(ds.labels.ravel(), prot_vector(ds, prot), nonsens_space(X, ds, prot))
    return _FULL_CACHE[name]

def prepare(name, seed=SEED):
    ds, prot = get_dataset(name)
    s_all, y_all = prot_vector(ds, prot), ds.labels.ravel()
    tr, te = train_test_split(np.arange(len(y_all)), test_size=TEST_SIZE, random_state=seed,
                              stratify=(2 * y_all + s_all))
    ds_tr, ds_te = ds.subset(tr), ds.subset(te)
    scaler = MinMaxScaler().fit(ds_tr.features)                      # fit on train only
    ds_tr.features, ds_te.features = scaler.transform(ds_tr.features), scaler.transform(ds_te.features)
    return dict(name=name, prot=prot, ds_tr=ds_tr, ds_te=ds_te, y_tr=ds_tr.labels.ravel(), y_te=ds_te.labels.ravel(),
                s_te=prot_vector(ds_te, prot), Xn_te=nonsens_space(ds_te.features, ds_te, prot),
                n_all=len(y_all), n_train=len(tr), n_test=len(te), n_features=ds.features.shape[1])

def reweigh(P):
    prot = P["prot"]
    rw = Reweighing(unprivileged_groups=[{prot: 0}], privileged_groups=[{prot: 1}])
    return rw.fit_transform(P["ds_tr"]).instance_weights.copy()

def _features(P, key):
    X = P[key].features
    return X if USE_SENSITIVE_AS_FEATURE else nonsens_space(X, P[key], P["prot"])

def _label(base, cname):
    return base if len(CLASSIFIERS) == 1 else f"{base} ({cname})"

def run_dataset(name, seed=SEED, include_full=True, verbose=True):
    P = prepare(name, seed)
    X_tr, X_te, y_tr, y_te = _features(P, "ds_tr"), _features(P, "ds_te"), P["y_tr"], P["y_te"]
    w = reweigh(P)
    rows = {}
    if include_full:
        rows["Original labels (full data)"] = full_data_report(name)
    rows["Original labels (test split)"] = fairness_report(y_te, P["s_te"], P["Xn_te"])
    for cname, make in CLASSIFIERS.items():
        for base, weights in (("Baseline classifier", None), ("Reweighing + classifier", w)):
            clf = make()
            clf.fit(X_tr, y_tr, sample_weight=weights)
            pred = clf.predict(X_te)
            rows[_label(base, cname)] = {**fairness_report(pred, P["s_te"], P["Xn_te"]),
                                         **equalized_odds_diff(y_te, pred, P["s_te"]),
                                         **performance_report(y_te, pred)}
    res = pd.DataFrame(rows).T
    res.insert(0, "Dataset", name)
    res.insert(1, "Seed", seed)
    if verbose:
        print(f"{name} (seed {seed}): n={P['n_all']}, features={P['n_features']}, train={P['n_train']}, test={P['n_test']}")
    return res

## 6. What the weights look like
For each combination of group `s` and label `y` the weight is `P(s)·P(y) / P(s, y)`: *expected* frequency if `s` and `y` were independent divided by the *observed* frequency. Cells that are rarer than independence would predict (typically unprivileged & favourable, privileged & unfavourable) get weight > 1, over-represented cells get weight < 1. The cell below recomputes the weights by hand, checks that they match AIF360's, and shows that the weighted training labels have DP_Diff ≈ 0 by construction.

In [7]:
def weights_table(name, seed=SEED):
    P = prepare(name, seed)
    y, s, w = P["y_tr"], prot_vector(P["ds_tr"], P["prot"]), reweigh(P)
    rows = []
    for sv, slab in ((0, "unprivileged"), (1, "privileged")):
        for yv, ylab in ((1, "favourable"), (0, "unfavourable")):
            m = (s == sv) & (y == yv)
            observed, expected = m.mean(), (s == sv).mean() * (y == yv).mean()
            rows.append(dict(Dataset=name, Group=slab, Outcome=ylab, Count=int(m.sum()), Observed=observed,
                             Expected=expected, Weight_formula=expected / observed, Weight_AIF360=w[m][0]))
    t = pd.DataFrame(rows)
    assert np.allclose(t["Weight_formula"], t["Weight_AIF360"]), "hand-computed weights differ from AIF360"
    dp_before = y[s == 1].mean() - y[s == 0].mean()
    dp_after = np.average(y[s == 1], weights=w[s == 1]) - np.average(y[s == 0], weights=w[s == 0])
    return t, dp_before, dp_after

wt = []
for name in DATASETS:
    t, b, a = weights_table(name)
    wt.append(t)
    print(f"{name}: DP_Diff of the training labels  before = {b:+.3f}   after weighting = {a:+.3f}")
    show(t.drop(columns=["Dataset"]).round(3))
pd.concat(wt).round(4).to_csv("reweighing_weights.csv", index=False)

German: DP_Diff of the training labels  before = +0.073   after weighting = +0.000


,Group,Outcome,Count,Observed,Expected,Weight_formula,Weight_AIF360
0,unprivileged,favourable,141,0.201,0.217,1.077,1.077
1,unprivileged,unfavourable,76,0.109,0.093,0.857,0.857
2,privileged,favourable,349,0.499,0.483,0.969,0.969
3,privileged,unfavourable,134,0.191,0.207,1.081,1.081


COMPAS: DP_Diff of the training labels  before = +0.098   after weighting = +0.000


,Group,Outcome,Count,Observed,Expected,Weight_formula,Weight_AIF360
0,unprivileged,favourable,1457,0.337,0.359,1.065,1.065
1,unprivileged,unfavourable,1391,0.322,0.300,0.932,0.932
2,privileged,favourable,897,0.208,0.186,0.894,0.894
3,privileged,unfavourable,575,0.133,0.155,1.165,1.165


Adult: DP_Diff of the training labels  before = +0.199   after weighting = +0.000


,Group,Outcome,Count,Observed,Expected,Weight_formula,Weight_AIF360
0,unprivileged,favourable,1168,0.037,0.081,2.183,2.183
1,unprivileged,unfavourable,9118,0.288,0.244,0.849,0.849
2,privileged,favourable,6677,0.211,0.167,0.793,0.793
3,privileged,unfavourable,14692,0.464,0.508,1.094,1.094


## 7. Run all datasets over all splits

In [8]:
def run_everything(with_s):
    global USE_SENSITIVE_AS_FEATURE
    previous, USE_SENSITIVE_AS_FEATURE = USE_SENSITIVE_AS_FEATURE, with_s
    try:
        out = []
        for name in DATASETS:
            t0 = time.time()
            for s in EVAL_SEEDS[name]:
                out.append(run_dataset(name, seed=s, include_full=(s == SEED), verbose=(s == SEED)))
            print(f"   {name}: {len(EVAL_SEEDS[name])} splits in {time.time() - t0:.0f}s")
        out = pd.concat(out)
        out.index.name = "Setting"
        return out
    finally:
        USE_SENSITIVE_AS_FEATURE = previous

print("=== MAIN experiment: protected attribute is a classifier input =", USE_SENSITIVE_AS_FEATURE)
runs = run_everything(USE_SENSITIVE_AS_FEATURE)
runs.reset_index().to_csv("results_reweighing_multiseed.csv", index=False)

runs_noS = None
if RUN_A2_VARIANT:
    print("=== VARIANT (paper's assumption A2): protected attribute is NOT a classifier input")
    runs_noS = run_everything(not USE_SENSITIVE_AS_FEATURE)
    runs_noS.reset_index().to_csv("results_reweighing_variant_multiseed.csv", index=False)

=== MAIN experiment: protected attribute is a classifier input = True
German (seed 42): n=1000, features=58, train=700, test=300
   German: 10 splits in 0s
COMPAS (seed 42): n=6172, features=13, train=4320, test=1852
   COMPAS: 10 splits in 1s
Adult (seed 42): n=45222, features=102, train=31655, test=13567
   Adult: 10 splits in 17s
=== VARIANT (paper's assumption A2): protected attribute is NOT a classifier input
German (seed 42): n=1000, features=58, train=700, test=300
   German: 10 splits in 1s
COMPAS (seed 42): n=6172, features=13, train=4320, test=1852
   COMPAS: 10 splits in 1s
Adult (seed 42): n=45222, features=102, train=31655, test=13567
   Adult: 10 splits in 14s


### Headline result (seed 42 test split) – before vs after

In [9]:
cols = ["DP_Diff", "DIR", "Consistency", "AEO_Diff", "Accuracy", "BalancedAcc", "F1"]
headline = runs[runs["Seed"] == SEED].reset_index()[["Dataset", "Setting"] + cols]
headline.round(3).to_csv("results_reweighing.csv", index=False)
show(headline.round(3))

,Dataset,Setting,DP_Diff,DIR,Consistency,AEO_Diff,Accuracy,BalancedAcc,F1
0,German,Original labels (full data),0.075,0.897,0.670,NaN,NaN,NaN,NaN
1,German,Original labels (test split),0.079,0.890,0.642,NaN,NaN,NaN,NaN
2,German,Baseline classifier,0.078,0.906,0.792,0.071,0.767,0.671,0.845
3,German,Reweighing + classifier,-0.025,1.031,0.771,-0.048,0.767,0.675,0.844
4,COMPAS,Original labels (full data),0.097,0.840,0.627,NaN,NaN,NaN,NaN
5,COMPAS,Original labels (test split),0.097,0.841,0.610,NaN,NaN,NaN,NaN
6,COMPAS,Baseline classifier,0.196,0.734,0.963,0.181,0.666,0.658,0.710
7,COMPAS,Reweighing + classifier,0.018,0.971,0.909,0.001,0.659,0.650,0.706
8,Adult,Original labels (full data),0.199,0.363,0.792,NaN,NaN,NaN,NaN
9,Adult,Original labels (test split),0.199,0.364,0.786,NaN,NaN,NaN,NaN


### Stability over random splits (mean ± std over the seeds in `EVAL_SEEDS`)

In [10]:
def mean_std(df, cols):
    g = df.reset_index().groupby(["Dataset", "Setting"], sort=False)[cols]
    m, s, n = g.mean(), g.std(), g.size()
    out = m.copy().astype(object)
    for c in cols:
        out[c] = ["–" if pd.isna(a) else (f"{a:.3f} ± {b:.3f}" if pd.notna(b) else f"{a:.3f}") for a, b in zip(m[c], s[c])]
    out.insert(0, "n_splits", n)
    return out

summary = mean_std(runs, cols)
summary.to_csv("results_reweighing_summary.csv")
show(summary)

n_splits        DP_Diff            DIR    Consistency        AEO_Diff       Accuracy    BalancedAcc             F1
Dataset Setting                                                                                                                                         
German  Original labels (full data)          1          0.075          0.897          0.670               –              –              –              –
        Original labels (test split)        10  0.079 ± 0.000  0.890 ± 0.000  0.648 ± 0.010               –              –              –              –
        Baseline classifier                 10  0.124 ± 0.057  0.848 ± 0.071  0.785 ± 0.019   0.110 ± 0.052  0.749 ± 0.021  0.663 ± 0.029  0.831 ± 0.014
        Reweighing + classifier             10  0.055 ± 0.071  0.932 ± 0.088  0.779 ± 0.022   0.027 ± 0.077  0.750 ± 0.017  0.664 ± 0.026  0.831 ± 0.012
COMPAS  Original labels (full data)          1          0.097          0.840          0.627               –              –              –              –
        Original labels (test split)        10  0.097 ± 0.000  0.841 ± 0.000  0.619 ± 0.008               –              –              –              –
        Baseline classifier                 10  0.211 ± 0.015  0.721 ± 0.015  0.969 ± 0.003   0.192 ± 0.015  0.675 ± 0.008  0.666 ± 0.008  0.720 ± 0.008
        Reweighing + classifier             10  0.034 ± 0.018  0.948 ± 0.025  0.914 ± 0.005   0.010 ± 0.018  0.672 ± 0.008  0.663 ± 0.008  0.720 ± 0.007
Adult   Original labels (full data)          1          0.199          0.363          0.792               –              –              –              –
        Original labels (test split)        10  0.199 ± 0.000  0.364 ± 0.000  0.786 ± 0.003               –              –              –              –
        Baseline classifier                 10  0.188 ± 0.005  0.282 ± 0.014  0.940 ± 0.001   0.105 ± 0.010  0.848 ± 0.003  0.764 ± 0.004  0.660 ± 0.007
        Reweighing + classifier             10  0.085 ± 0.004  0.599 ± 0.016  0.939 ± 0.002  -0.068 ± 0.009  0.841 ± 0.003  0.745 ± 0.004  0.633 ± 0.006

### Variant: protected attribute NOT a classifier input (paper's assumption A2)

In [11]:
if runs_noS is not None:
    summary_noS = mean_std(runs_noS, cols)
    summary_noS.to_csv("results_reweighing_variant_summary.csv")
    show(summary_noS)

n_splits        DP_Diff            DIR    Consistency        AEO_Diff       Accuracy    BalancedAcc             F1
Dataset Setting                                                                                                                                         
German  Original labels (full data)          1          0.075          0.897          0.670               –              –              –              –
        Original labels (test split)        10  0.079 ± 0.000  0.890 ± 0.000  0.648 ± 0.010               –              –              –              –
        Baseline classifier                 10  0.064 ± 0.061  0.920 ± 0.075  0.779 ± 0.020   0.041 ± 0.063  0.755 ± 0.020  0.670 ± 0.028  0.834 ± 0.014
        Reweighing + classifier             10  0.052 ± 0.062  0.935 ± 0.077  0.780 ± 0.020   0.025 ± 0.069  0.750 ± 0.019  0.664 ± 0.027  0.832 ± 0.013
COMPAS  Original labels (full data)          1          0.097          0.840          0.627               –              –              –              –
        Original labels (test split)        10  0.097 ± 0.000  0.841 ± 0.000  0.619 ± 0.008               –              –              –              –
        Baseline classifier                 10  0.198 ± 0.010  0.735 ± 0.010  0.971 ± 0.004   0.179 ± 0.011  0.674 ± 0.009  0.665 ± 0.009  0.720 ± 0.009
        Reweighing + classifier             10  0.197 ± 0.010  0.737 ± 0.010  0.972 ± 0.004   0.178 ± 0.011  0.675 ± 0.009  0.666 ± 0.009  0.721 ± 0.009
Adult   Original labels (full data)          1          0.199          0.363          0.792               –              –              –              –
        Original labels (test split)        10  0.199 ± 0.000  0.364 ± 0.000  0.786 ± 0.003               –              –              –              –
        Baseline classifier                 10  0.180 ± 0.005  0.301 ± 0.011  0.940 ± 0.001   0.090 ± 0.009  0.847 ± 0.003  0.762 ± 0.004  0.658 ± 0.007
        Reweighing + classifier             10  0.094 ± 0.004  0.566 ± 0.014  0.939 ± 0.001  -0.052 ± 0.010  0.842 ± 0.002  0.746 ± 0.004  0.635 ± 0.006

### LaTeX tables for the report
`write_latex_tables` turns `runs` into three ready-made tables (datasets; fairness and performance before/after; sensitivity check without the protected attribute) and writes them to `results_tables.tex`; the report includes this file with `\input{results_tables}`. Copy the generated file into the Overleaf project (it replaces the dummy version).

In [12]:
PROT_TEXT = {"German": ("good credit", "sex (male)"), "COMPAS": ("no recidivism", "race (Caucasian)"),
             "Adult": ("income $>$ 50K", "sex (male)")}
FAIR_COLS = [("DP_Diff", r"DP\_Diff"), ("DIR", "DIR"), ("Consistency", "Consistency"), ("AEO_Diff", r"AEO\_Diff")]
PERF_COLS = [("Accuracy", "Accuracy"), ("BalancedAcc", "Balanced acc."), ("F1", "F1")]
RES_COLS  = FAIR_COLS + PERF_COLS
VAR_COLS  = [("DP_Diff", r"DP\_Diff"), ("DIR", "DIR"), ("AEO_Diff", r"AEO\_Diff"), ("BalancedAcc", "Balanced acc.")]

def _num(x, nd=3):
    return "--" if x is None or pd.isna(x) else f"{x:.{nd}f}".replace("-", "$-$")

def write_latex_tables(runs=None, runs_variant=None, path="results_tables.tex", dummy=False):
    """dummy=True writes the same layout with placeholder values (-100 and -1000%)."""
    primary = next(iter(CLASSIFIERS))
    key_name = {"orig": "Original labels (full data)", "base": _label("Baseline classifier", primary),
                "rw": _label("Reweighing + classifier", primary)}
    n_splits = len(EVAL_SEEDS[DATASETS[0]])

    def make(df):
        df = None if dummy else df.reset_index()

        def stat(ds, key, col):
            """(mean, std) or None when the measure does not exist for that row."""
            if key == "orig" and col in ("AEO_Diff", "Accuracy", "BalancedAcc", "F1"):
                return None
            if dummy:
                return (-100.0, None) if key == "orig" else (-100.0, -100.0)
            v = df[(df["Dataset"] == ds) & (df["Setting"] == key_name[key])][col].dropna()
            return None if v.empty else (v.mean(), v.std() if len(v) > 1 else None)

        def cell(ds, key, col):
            st = stat(ds, key, col)
            if st is None:
                return "--"
            mean, std = st
            if dummy:
                return "-100" + ("" if std is None else r" $\pm$ -100")
            return _num(mean) + ("" if std is None or pd.isna(std) else r" $\pm$ " + _num(std))

        def change(ds, ref, col):
            a, b = stat(ds, "rw", col), stat(ds, ref, col)
            if a is None or b is None:
                return "--"
            if dummy:
                return r"-1000\%"
            if abs(b[0]) < 1e-3:
                return "n/a"
            return f"{int(round((a[0] - b[0]) / abs(b[0]) * 100))}\\%".replace("-", "$-$")

        def block(cols, rows_spec, ds):
            lines = []
            for j, (label, key, kind) in enumerate(rows_spec):
                first = (r"\multirow{%d}{*}{%s}" % (len(rows_spec), ds)) if j == 0 else ""
                vals = [cell(ds, key, c) if kind == "val" else change(ds, key, c) for c, _ in cols]
                lines.append(f"{first} & {label} & " + " & ".join(vals) + r" \\")
            return "\n".join(lines)
        return block

    def table(block, cols, rows_spec, caption, label, size=r"\footnotesize"):
        spec = "ll" + "r" * len(cols)
        head = " & ".join(r"\textbf{%s}" % h for _, h in cols)
        lines = [r"\begin{table}[ht]", r"\centering", r"{" + size, r"\setlength{\tabcolsep}{3pt}",
                 r"\begin{tabular}{%s}" % spec, r"\toprule", r"\textbf{Dataset} & \textbf{Setting} & " + head + r" \\"]
        for ds in DATASETS:
            lines += [r"\midrule", block(cols, rows_spec, ds)]
        lines += [r"\bottomrule", r"\end{tabular}", r"}", r"\caption{" + caption + "}", r"\label{" + label + "}",
                  r"\end{table}", ""]
        return "\n".join(lines)

    out = []
    # ---- Table 1: datasets and groups
    out.append(r"""\begin{table}[ht]
\centering
{\footnotesize
\setlength{\tabcolsep}{4pt}
\begin{tabular}{lllrrrr}
\toprule
\textbf{Dataset} & \textbf{Favourable} & \textbf{Protected (privileged)} & \textbf{Rows} & \textbf{Features} & \textbf{P(fav$\mid$priv)} & \textbf{P(fav$\mid$unpriv)} \\
\midrule""")
    for ds in DATASETS:
        fav, prot = PROT_TEXT[ds]
        if dummy:
            nums = ["-100"] * 4
        else:
            d, _ = get_dataset(ds); full = full_data_report(ds)
            nums = [f"{len(d.labels):,}".replace(",", r"\,"), str(d.features.shape[1]),
                    _num(full["P(fav|priv)"]), _num(full["P(fav|unpriv)"])]
        out.append(f"{ds} & {fav} & {prot} & " + " & ".join(nums) + r" \\")
    out.append(r"""\bottomrule
\end{tabular}
}
\caption{Datasets and groups. Rows and features (after one-hot encoding, including the protected attribute) are counted after pre-processing; P(fav$\mid$priv) and P(fav$\mid$unpriv): share of favourable original labels per group.}
\label{tab:datasets}
\end{table}
""")
    main = make(runs)
    rows_f = [("Original labels", "orig", "val"), ("Baseline classifier", "base", "val"), ("Reweighing", "rw", "val"),
              (r"$\Delta$ vs.\ original (\%)", "orig", "chg"), (r"$\Delta$ vs.\ baseline (\%)", "base", "chg")]
    out.append(table(main, RES_COLS, rows_f,
        r"Fairness and performance. Original labels: whole dataset, before mitigation; other rows: mean $\pm$ standard deviation on the held-out test split over " + str(n_splits) +
        r" stratified splits (baseline = logistic regression without weights). DP\_Diff, AEO\_Diff: privileged minus unprivileged (optimum 0); DIR: unprivileged over privileged (optimum 1); consistency: $k=5$ (optimum 1). $\Delta$ = (Reweighing $-$ reference)/$|$reference$|$; n/a: reference almost 0; --: undefined.",
        "tab:results", size=r"\scriptsize"))
    if dummy or runs_variant is not None:
        var = make(runs_variant)
        rows_v = [("Baseline classifier", "base", "val"), ("Reweighing", "rw", "val")]
        out.append(table(var, VAR_COLS, rows_v,
            r"Sensitivity check (paper's assumption A2): the protected attribute is used for the weights and the evaluation but is \emph{not} a classifier input (mean $\pm$ standard deviation over " + str(n_splits) + r" splits).",
            "tab:variant"))
    with open(path, "w", encoding="utf-8") as f:
        f.write("% Generated by write_latex_tables() -- " + ("DUMMY VALUES" if dummy else "real results") +
                "\n% Needs \\usepackage{booktabs} and \\usepackage{multirow} in main.tex\n\n" + "\n".join(out))
    print(f"wrote {path} ({'dummy values' if dummy else 'real results'})")

write_latex_tables(runs, runs_noS, "results_tables.tex")

wrote results_tables.tex (real results)


In [13]:
import platform, sklearn, aif360
print("Environment (copy into the report):",
      f"Python {platform.python_version()}, scikit-learn {sklearn.__version__}, pandas {pd.__version__}, "
      f"numpy {np.__version__}, AIF360 {aif360.__version__}, {platform.system()} {platform.release()}")
print(f"Total run time: {(time.time() - T_START) / 60:.1f} min")

Environment (copy into the report): Python 3.13.9, scikit-learn 1.9.1, pandas 2.3.3, numpy 2.5.4, AIF360 0.6.1, Windows 11
Total run time: 0.6 min
